# Authentification
### On s'authentifie avec notre compte Google lié
### au projet Cloud 'stress-hydrique-491820'
### force=True : force une nouvelle authentification
### même si des credentials existent déjà en cache

In [ ]:
import ee

ee.Authenticate(force=True)
ee.Initialize(project='stress-hydrique-491820')

print("✅ Connexion GEE réussie !")

✅ Connexion GEE réussie !


# Définition de la zone d'étude
### On travaille sur un point précis dans la région
### Mandoul (sud du Tchad) — zone agricole mil/sorgho
### Coordonnées : [longitude, latitude]
### On commence par un point (pas tout le pays)
### pour limiter le volume de données traitées



In [ ]:
zone_tchad = ee.Geometry.Point([17.4, 8.6])

print("Zone définie :", zone_tchad.getInfo())

Zone définie : {'type': 'Point', 'coordinates': [17.4, 8.6]}


# Chargement collection Sentinel-2
### Dataset : COPERNICUS/S2_SR_HARMONIZED (L2A)
### = images corrigées atmosphériquement

### Filtres appliqués :
###   - filterBounds : uniquement notre zone Mandoul
###   - filterDate : saison des pluies juin-sept 2023
###     (période de croissance mil/sorgho au Tchad)
###   - CLOUDY_PIXEL_PERCENTAGE < 50% : on accepte
###     jusqu'à 50% de nuages (saison des pluies)

### Masque nuages :
###   La bande SCL (Scene Classification Layer) classe
###   chaque pixel. SCL=9 = nuage. On met ces pixels
###   à zéro pour ne pas fausser nos indices NDVI/NDWI


In [ ]:
def masquer_nuages(image):
    masque = image.select('SCL').neq(9)  # SCL=9 = nuage → masqué
    return image.updateMask(masque)

collection_propre = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(zone_tchad) \
    .filterDate('2023-06-01', '2023-09-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)) \
    .map(masquer_nuages)

print("Images disponibles :", collection_propre.size().getInfo())

Images disponibles : 5


# Calcul du NDVI sur première image
### NDVI = (B8 - B4) / (B8 + B4)
###   B8 = Proche Infrarouge (plante saine → réfléchit)
###   B4 = Rouge             (plante saine → absorbe)

### Interprétation :
###   NDVI ≈ 0.8-1.0 → végétation dense et saine
###   NDVI ≈ 0.3-0.6 → végétation modérée ou stressée
###   NDVI ≈ 0.0-0.2 → sol nu ou végétation très faible
###   NDVI < 0.0     → eau, nuages résiduels

### reduceRegion : calcule la moyenne NDVI
### sur tous les pixels dans notre zone (scale=10m)

In [ ]:
premiere_image = collection_propre.first()

ndvi = premiere_image.normalizedDifference(['B8', 'B4']).rename('NDVI')

valeur_ndvi = ndvi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=zone_tchad,
    scale=10
).getInfo()

print("Valeur NDVI sur notre zone :", valeur_ndvi)

Valeur NDVI sur notre zone : {'NDVI': 0.8051787916152897}
